# 🏢 Urban Economics: Paris & Lyon - Monocentric or Polycentric?

**Authors:** Jérémie Konda & Alexandre Klobb
**Date:** April 11, 2025 (Analysis Date: *Insert Current Date*)

## 🎯 Objective

This notebook investigates the evolution of the spatial distribution of employment in the French metropolitan areas of **Paris** and **Lyon** between **1968 and 2021**. The primary goal is to determine whether these cities still adhere to a **monocentric** model, as described by Alonso (1964), or if they have transitioned towards a **polycentric** structure.

We will use employment data from INSEE and geographical data from IGN, applying concepts from urban economics like the bid-rent theory (Alonso's model), transport costs, and employment concentration.

**Theoretical Background:**

*   **Monocentric Model (Alonso, 1964):** Assumes a single central business district (CBD) where most jobs are located. Households trade off commuting costs against housing costs (rents decrease with distance from the CBD). This predicts a negative gradient for population density and land rents moving away from the center.
*   **Polycentrism:** Acknowledges the emergence of secondary employment centers (subcenters) outside the main CBD, leading to a more complex urban structure and potentially flatter or multi-peaked rent/density gradients.

In [ ]:
# Setup: Import Libraries and Define Constants

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import statsmodels.formula.api as smf
import requests
import os
import py7zr # Used for extracting map data if needed
from shapely.geometry import Point
import warnings

# Ignore warnings for cleaner output (optional)
warnings.filterwarnings('ignore')

# Set a visual style for plots
sns.set_theme(style="whitegrid")

# --- Configuration ---

# Define base directories relative to the notebook location
# Assumes the notebook is in the root folder, alongside 'input' and 'output'
REPERTOIRE_ENTREE = "input"
REPERTOIRE_SORTIE = "output"

# Define periods for analysis
PERIODES = [1968, 1999, 2021]

# Define file paths
CHEMIN_EMPLOI = os.path.join(REPERTOIRE_ENTREE, "pop-act2554-empl-csp-cd-trav-6821_lieu-travail.xlsx")
CHEMIN_ZONES = os.path.join(REPERTOIRE_ENTREE, "AU2010_au_01-01-2020.xlsx")

# Geographical data (Shapefile) - ASSUMES MANUAL DOWNLOAD AND EXTRACTION
# User should download from the URL and extract the SHP file into REPERTOIRE_ENTREE/carte/
# URL: https://data.geopf.fr/telechargement/download/GEOFLA/GEOFLA_2-2_COMMUNE_SHP_LAMB93_FXX_2016-06-28/GEOFLA_2-2_COMMUNE_SHP_LAMB93_FXX_2016-06-28.7z
CARTE_DIR = os.path.join(REPERTOIRE_ENTREE, "carte")
# Try to automatically find the SHP file within the carte directory
SHP_FILE_PATH = None
if os.path.exists(CARTE_DIR):
    for root, dirs, files in os.walk(CARTE_DIR):
        for file in files:
            if file.lower().endswith('.shp'):
                SHP_FILE_PATH = os.path.join(root, file)
                break
        if SHP_FILE_PATH:
            break

# Create output directories if they don't exist
os.makedirs(os.path.join(REPERTOIRE_SORTIE, "img/carte"), exist_ok=True)
os.makedirs(os.path.join(REPERTOIRE_SORTIE, "Paris"), exist_ok=True)
os.makedirs(os.path.join(REPERTOIRE_SORTIE, "Lyon"), exist_ok=True)


print("Libraries imported and constants defined.")
print(f"Employment data expected at: {CHEMIN_EMPLOI}")
print(f"Urban area data expected at: {CHEMIN_ZONES}")
if SHP_FILE_PATH:
    print(f"Shapefile automatically found at: {SHP_FILE_PATH}")
else:
    print(f"Shapefile NOT FOUND in {CARTE_DIR}. Please download and extract it manually from the URL mentioned in the comments.")

## Data Loading and Preparation

### 1. Employment Data (INSEE)

We load employment data for French communes from an Excel file provided by INSEE. This file contains employment counts by socio-professional category for the years 1968, 1999, and 2021 (among others).

**Steps:**
1.  Read the relevant sheets (`COM_1968`, `COM_1999`, `COM_2021`) from the Excel file.
2.  Skip initial metadata rows.
3.  Filter out irrelevant rows (e.g., regional totals 'RLT').
4.  Calculate the total employment (`Total_Emploi_YYYY`) for each commune by summing across categories.
5.  Create a standardized 5-digit INSEE code (`code`) for merging with other datasets.

In [ ]:
def preparer_donnees_emploi(chemin_fichier, annees):
    """Loads and prepares employment data for specified years from INSEE file."""
    donnees_par_annee = {}
    try:
        fichier_excel = pd.ExcelFile(chemin_fichier)
    except FileNotFoundError:
        print(f"ERREUR: Fichier d'emploi non trouvé à {chemin_fichier}")
        return None

    print(f"Loading employment data for years: {annees}")
    for annee in annees:
        sheet_name = f'COM_{annee}'
        if sheet_name in fichier_excel.sheet_names:
            print(f" Reading sheet: {sheet_name}")
            try:
                df = pd.read_excel(fichier_excel, sheet_name, skiprows=13)

                # Check if essential columns exist
                region_col = 'Région \nen géographie courante'
                dept_col = "Département\nen géographie courante"
                comm_col = "Commune\nen géographie courante"
                required_data_cols = [
                    f"Agriculteurs\nRP{annee}",
                    f"Artisans, commerçants, chefs d'entreprises\nRP{annee}",
                    f"Cadres et professions intellectuelles supérieures\nRP{annee}",
                    f"Professions intermédiaires\nRP{annee}",
                    f"Employés\nRP{annee}",
                    f"Ouvriers\nRP{annee}"
                ]

                if not all(col in df.columns for col in [region_col, dept_col, comm_col] + required_data_cols):
                     print(f" AVERTISSEMENT: Colonnes manquantes dans la feuille {sheet_name}. Skipping.")
                     continue

                # Suppression des lignes non pertinentes
                df = df[df[region_col] != 'RLT']
                df = df.dropna(subset=[dept_col, comm_col]) # Drop rows where commune/dept code is missing

                # Calcul du total des emplois
                df[f'Total_Emploi_{annee}'] = df[required_data_cols].sum(axis=1)

                # Création du code INSEE standardisé (5 digits)
                # Convert department and commune codes to strings, handling potential non-numeric values
                df[dept_col] = df[dept_col].astype(str).str.split('.').str[0] # Handle potential floats like '2A.0'
                df[comm_col] = df[comm_col].astype(str).str.split('.').str[0]

                # Pad with zeros
                df['code'] = (
                    df[dept_col].str.zfill(2) +
                    df[comm_col].str.zfill(3)
                )

                # Select only relevant columns for this year
                donnees_par_annee[annee] = df[['code', f'Total_Emploi_{annee}']].copy()

            except Exception as e:
                print(f" Erreur lors de la lecture de la feuille {sheet_name}: {e}")
        else:
            print(f" AVERTISSEMENT: Feuille {sheet_name} non trouvée dans le fichier Excel.")

    # Combine data from all years into a single DataFrame
    if not donnees_par_annee:
        print("ERREUR: Aucune donnée d'emploi n'a pu être chargée.")
        return None

    # Start with the first available year's data
    first_year = min(donnees_par_annee.keys())
    donnees_combinees = donnees_par_annee[first_year]

    # Merge subsequent years
    for year in sorted(donnees_par_annee.keys()):
        if year != first_year:
            donnees_combinees = pd.merge(
                donnees_combinees,
                donnees_par_annee[year],
                on='code',
                how='outer' # Keep all communes from all years
            )

    # Fill missing employment values with 0, assuming commune existed but had 0 jobs reported or wasn't reported
    for annee in annees:
         col_name = f'Total_Emploi_{annee}'
         if col_name in donnees_combinees.columns:
             donnees_combinees[col_name] = donnees_combinees[col_name].fillna(0)

    # Add commune names and department from one of the sheets for context (e.g., 2021)
    try:
        df_ref = pd.read_excel(fichier_excel, f'COM_{max(annees)}', skiprows=13)
        df_ref = df_ref[df_ref['Région \nen géographie courante'] != 'RLT']
        df_ref[dept_col] = df_ref["Département\nen géographie courante"].astype(str).str.split('.').str[0]
        df_ref[comm_col] = df_ref["Commune\nen géographie courante"].astype(str).str.split('.').str[0]
        df_ref['code'] = (
            df_ref[dept_col].str.zfill(2) +
            df_ref[comm_col].str.zfill(3)
        )
        # Add Libelle column if it exists
        libelle_col = 'Libellé \nen géographie courante'
        if libelle_col in df_ref.columns:
             donnees_combinees = pd.merge(
                 donnees_combinees,
                 df_ref[['code', dept_col, libelle_col]].rename(columns={libelle_col: 'Libelle'}),
                 on='code',
                 how='left'
             )
        else: # Fallback if Libelle column name changed
             donnees_combinees = pd.merge(
                 donnees_combinees,
                 df_ref[['code', dept_col]].drop_duplicates(subset=['code']),
                 on='code',
                 how='left'
             )


    except Exception as e:
        print(f" AVERTISSEMENT: Impossible d'ajouter les libellés des communes. Erreur: {e}")


    print(f"Données d'emploi préparées. Nombre total de communes chargées: {len(donnees_combinees)}")
    return donnees_combinees

# Run the function
donnees_emploi_totales = preparer_donnees_emploi(CHEMIN_EMPLOI, PERIODES)

# Display a sample of the prepared data
if donnees_emploi_totales is not None:
    print("\nSample of prepared employment data:")
    display(donnees_emploi_totales.head())
    # Check for duplicates in 'code'
    print(f"\nNumber of duplicate INSEE codes: {donnees_emploi_totales.duplicated(subset=['code']).sum()}")
    # Display info
    print("\nDataFrame Info:")
    donnees_emploi_totales.info()
else:
    print("\nEmployment data could not be loaded.")

### 2. Urban Area Definitions (INSEE)

We use the INSEE definition of "Aires d'Attraction des Villes" (functional urban areas, AU2010 standard used here as per original script) to identify the communes belonging to the Paris and Lyon metropolitan areas.

**Steps:**
1.  Load the composition file.
2.  Filter for the codes corresponding to Paris (`001`) and Lyon (`002`).
3.  Extract the INSEE codes (`CODGEO`) for communes within these areas.
4.  Standardize the INSEE code format to 5 digits (`code`).

In [ ]:
def filtrer_agglomeration(donnees_emploi, chemin_zones_urbaines):
    """Filters employment data for Paris and Lyon urban areas."""
    if donnees_emploi is None:
        print("ERREUR: Les données d'emploi ne sont pas disponibles pour le filtrage.")
        return None, None

    try:
        zones_urbaines = pd.read_excel(chemin_zones_urbaines, 'Composition_communale', skiprows=5)
    except FileNotFoundError:
        print(f"ERREUR: Fichier des zones urbaines non trouvé à {chemin_zones_urbaines}")
        return None, None
    except Exception as e:
        print(f"ERREUR: Impossible de lire le fichier des zones urbaines : {e}")
        return None, None

    print("Filtering for Paris (AU2010 = 001) and Lyon (AU2010 = 002) urban areas...")

    # Filter for Lyon and Paris based on AU2010 code
    zones_lyon = zones_urbaines[zones_urbaines['AU2010'] == '002'].copy()
    zones_paris = zones_urbaines[zones_urbaines['AU2010'] == '001'].copy()

    if zones_lyon.empty or zones_paris.empty:
         print("AVERTISSEMENT: Impossible de trouver les zones pour Paris ou Lyon dans le fichier.")
         # Attempt filtering based on libelle if code fails
         zones_lyon = zones_urbaines[zones_urbaines['LIBAU2010'].str.contains("Lyon", case=False, na=False)].copy()
         zones_paris = zones_urbaines[zones_urbaines['LIBAU2010'].str.contains("Paris", case=False, na=False)].copy()
         if zones_lyon.empty or zones_paris.empty:
             print("ERREUR: Filtrage par libellé a également échoué.")
             return None, None
         else:
              print(" Filtrage par libellé réussi.")


    # Prepare codes for merging (ensure 5 digits)
    zones_lyon['code'] = zones_lyon['CODGEO'].astype(str).str.zfill(5)
    zones_paris['code'] = zones_paris['CODGEO'].astype(str).str.zfill(5)

    # Merge with employment data
    # Use 'inner' merge to keep only communes present in both employment data AND urban area definition
    paris_df = donnees_emploi.merge(zones_paris[['code']], on='code', how='inner')
    lyon_df = donnees_emploi.merge(zones_lyon[['code']], on='code', how='inner')

    # Specific fix: Add Paris arrondissements (codes 75101 to 75120) which might be missed
    # arrondissements_paris = donnees_emploi[donnees_emploi['code'].str.startswith('751')]
    # We should use the main Paris code '75056' for the whole city before merging if arrondissements aren't needed separately.
    # However, the original script logic suggests keeping arrondissements. Let's find them if they exist.
    arr_codes = [f"751{str(i).zfill(2)}" for i in range(1, 21)]
    arrondissements_paris = donnees_emploi[donnees_emploi['code'].isin(arr_codes)]

    # Combine the main urban area communes with the arrondissements if they exist
    if not arrondissements_paris.empty:
         # Ensure no duplicates if an arrondissement was somehow also in the main AU filter
         paris_df = pd.concat([paris_df, arrondissements_paris], ignore_index=True).drop_duplicates(subset=['code'])


    print(f"Filtered data: {len(paris_df)} communes in Paris area, {len(lyon_df)} communes in Lyon area.")

    # Check if dataframes are empty
    if paris_df.empty:
        print("AVERTISSEMENT: Le DataFrame de Paris est vide après le filtrage.")
    if lyon_df.empty:
        print("AVERTISSEMENT: Le DataFrame de Lyon est vide après le filtrage.")

    return paris_df, lyon_df

# Run the filtering function
paris_df, lyon_df = filtrer_agglomeration(donnees_emploi_totales, CHEMIN_ZONES)

# Display samples
if paris_df is not None:
    print("\nSample of Paris Area Data:")
    display(paris_df.head())
if lyon_df is not None:
    print("\nSample of Lyon Area Data:")
    display(lyon_df.head())


### 3. Geographical Data (IGN - GEOFLA® Communes)

We need the geographical boundaries of the communes to create maps and calculate distances from the city center. We use the GEOFLA® dataset from IGN.

**Manual Step Required:**
*   **Download:** Download the data from [this URL](https://data.geopf.fr/telechargement/download/GEOFLA/GEOFLA_2-2_COMMUNE_SHP_LAMB93_FXX_2016-06-28/GEOFLA_2-2_COMMUNE_SHP_LAMB93_FXX_2016-06-28.7z).
*   **Extract:** Extract the archive. You should find a `.shp` file (e.g., `COMMUNE.shp`) along with other related files (`.dbf`, `.shx`, etc.).
*   **Place:** Place all extracted files into the `input/carte/` directory within your project.

**Code Steps:**
1.  Check if the Shapefile (`.shp`) exists in the specified directory (`input/carte/`).
2.  Load the Shapefile using GeoPandas.
3.  Standardize the INSEE code column (`INSEE_COM`) to match our 5-digit `code` format.
4.  Handle specific historical changes: Merge the former communes of Lyon (69381-69389) into the main Lyon code (`69123` seems incorrect, usually it's `69001`-`69009` or aggregated into `69123` for the *arrondissements* within Lyon city proper `69000`. The original code used `69123` which corresponds to Lyon city historically. Let's keep it but note this might need adjustment based on exact analysis needs. INSEE code for Lyon *city* is `69123`). The code `69381`-`69389` refers to Lyon's arrondissements before 1964. Let's use `69123` as the target code representing the central entity for mapping if those old codes appear in the shapefile. More commonly, shapefiles use `75056` for Paris City and `69123` for Lyon City, and potentially `75101`-`75120` / `69381`-`69389` for arrondissements. We'll standardize the commune code and handle the Lyon merge as in the original script.

In [ ]:
def charger_donnees_geographiques(shp_file_path):
    """Loads commune boundaries from a Shapefile."""
    if shp_file_path is None or not os.path.exists(shp_file_path):
        print(f"ERREUR: Fichier SHP non trouvé. Vérifiez le chemin : {shp_file_path}")
        print("Veuillez télécharger et extraire les données GEOFLA dans le dossier 'input/carte/'")
        return None

    print(f"Loading geographical data from: {shp_file_path}")
    try:
        communes_gdf = gpd.read_file(shp_file_path)

        # Standardize CRS to a projected CRS suitable for distance calculation (like Lambert 93 - EPSG:2154)
        # The GEOFLA data is often already in Lambert 93
        if communes_gdf.crs is None:
            print(" AVERTISSEMENT: CRS non défini pour le shapefile. En supposant Lambert 93 (EPSG:2154).")
            communes_gdf.set_crs(epsg=2154, inplace=True)
        elif communes_gdf.crs.to_epsg() != 2154:
            print(f" Reprojection des données géographiques vers EPSG:2154 (Lambert 93) depuis {communes_gdf.crs.name}")
            communes_gdf = communes_gdf.to_crs(epsg=2154)

        # Standardize the INSEE code column
        if 'INSEE_COM' in communes_gdf.columns:
            communes_gdf['code'] = communes_gdf['INSEE_COM'].astype(str).str.zfill(5)
        elif 'code_insee' in communes_gdf.columns: # Handle alternative column names
             communes_gdf['code'] = communes_gdf['code_insee'].astype(str).str.zfill(5)
        else:
             print("ERREUR: Colonne de code INSEE ('INSEE_COM' ou 'code_insee') non trouvée dans le shapefile.")
             return None

        # Handle Lyon arrondissements (merge old codes 69381-69389 into 69123 if they exist)
        # These codes represent arrondissements within Lyon city.
        lyon_arr_codes = [str(c) for c in range(69381, 69390)] # '69381' to '69389'
        communes_gdf.loc[communes_gdf['code'].isin(lyon_arr_codes), 'code'] = '69123' # Map to Lyon city code

        # Handle Paris arrondissements similarly (merge 75101-75120 into 75056 - Paris city code)
        paris_arr_codes = [f"751{str(i).zfill(2)}" for i in range(1, 21)]
        communes_gdf.loc[communes_gdf['code'].isin(paris_arr_codes), 'code'] = '75056' # Map to Paris city code


        # Calculate centroids needed for distance calculations
        # Ensure geometry is valid before calculating centroid
        communes_gdf = communes_gdf[communes_gdf.geometry.is_valid]
        communes_gdf['centroid'] = communes_gdf.geometry.centroid
        communes_gdf['X_CENTROID'] = communes_gdf.centroid.x
        communes_gdf['Y_CENTROID'] = communes_gdf.centroid.y

        # Keep relevant columns (added NOM_COM for better readability if available)
        relevant_cols = ['code', 'geometry', 'X_CENTROID', 'Y_CENTROID']
        if 'NOM_COM' in communes_gdf.columns:
            relevant_cols.insert(1, 'NOM_COM')
        elif 'nom' in communes_gdf.columns:
             relevant_cols.insert(1, 'nom') # Handle alternative name column
             communes_gdf.rename(columns={'nom': 'NOM_COM'}, inplace=True)
        else:
             # Add a placeholder if no name column found
             communes_gdf['NOM_COM'] = 'N/A'
             relevant_cols.insert(1, 'NOM_COM')

        print(f"Données géographiques chargées et préparées pour {len(communes_gdf)} communes.")
        return communes_gdf[relevant_cols]

    except Exception as e:
        print(f"ERREUR lors du chargement ou traitement des données géographiques : {e}")
        return None

# Run the function
communes_gdf = charger_donnees_geographiques(SHP_FILE_PATH)

# Display sample
if communes_gdf is not None:
    print("\nSample of Geographical Data (GeoDataFrame):")
    display(communes_gdf.head())
    print("\nCoordinate Reference System (CRS):")
    print(communes_gdf.crs)


### 4. Merging Employment and Geographical Data

Now, we merge the filtered employment data (for Paris and Lyon separately) with the geographical data. This allows us to map employment figures onto the commune boundaries.

**Steps:**
1.  Perform a `merge` (like a SQL join) between the GeoDataFrame (`communes_gdf`) and the employment DataFrames (`paris_df`, `lyon_df`).
2.  The merge is done using the common `code` (INSEE code) column.
3.  We use `how='inner'` to keep only the communes that are present in *both* datasets (i.e., communes within the defined urban area that also have geographical boundaries available).

In [ ]:
def merge_data(gdf, df, city_name):
    """Merges geographical data with employment data."""
    if gdf is None or df is None:
        print(f"ERREUR: Données géographiques ou d'emploi manquantes pour {city_name}.")
        return None

    print(f"Merging geographical and employment data for {city_name}...")

    # Ensure 'code' columns are of the same type (string)
    gdf['code'] = gdf['code'].astype(str)
    df['code'] = df['code'].astype(str)

    # Perform the merge
    # Drop geometry temporarily from df if it exists to avoid conflict
    cols_to_merge = df.columns.difference(gdf.columns).tolist() + ['code']
    # Handle case where 'Libelle' might exist from earlier step
    if 'Libelle' in df.columns and 'Libelle' not in gdf.columns:
        if 'Libelle' not in cols_to_merge: cols_to_merge.append('Libelle')
    
    merged_gdf = gdf.merge(df[cols_to_merge], on='code', how='inner')

    # Check for duplicates introduced by merge (shouldn't happen with 'inner' on unique 'code')
    if merged_gdf.duplicated(subset=['code']).any():
        print(f" AVERTISSEMENT: Duplicates found after merging for {city_name}. Check input data.")
        merged_gdf = merged_gdf.drop_duplicates(subset=['code'], keep='first')


    print(f"Merge successful for {city_name}. Resulting GeoDataFrame has {len(merged_gdf)} communes.")
    if merged_gdf.empty:
         print(f" AVERTISSEMENT: Le GeoDataFrame fusionné pour {city_name} est vide.")

    return merged_gdf

# Merge for Paris
paris_com_gdf = merge_data(communes_gdf, paris_df, "Paris")
# Merge for Lyon
lyon_com_gdf = merge_data(communes_gdf, lyon_df, "Lyon")

# Display samples of merged data
if paris_com_gdf is not None:
    print("\nSample of Merged Paris Data (GeoDataFrame):")
    display(paris_com_gdf.head())
    print(f"\nColumns: {paris_com_gdf.columns.tolist()}")

if lyon_com_gdf is not None:
    print("\nSample of Merged Lyon Data (GeoDataFrame):")
    display(lyon_com_gdf.head())
    print(f"\nColumns: {lyon_com_gdf.columns.tolist()}")


## Analysis: Paris

We now analyze the spatial distribution of employment in the Paris metropolitan area over the three periods (1968, 1999, 2021).

**Steps:**
1.  **Mapping:** Visualize the number of jobs per commune for each year.
2.  **Distance Calculation:** Calculate the distance of each commune's centroid from the defined center of Paris (Notre-Dame Cathedral).
3.  **Cumulative Distribution:** Plot the cumulative share of total employment as a function of distance from the center.
4.  **Employment-Distance Relationship:** Analyze the relationship between the logarithm of employment and the logarithm of distance using scatter plots and linear regression (OLS).

**Center Definition:**
*   **Paris Center:** Notre-Dame de Paris. We use its coordinates in the Lambert-93 projection (EPSG:2154).

In [ ]:
# Define Center for Paris (Notre-Dame de Paris in Lambert-93 / EPSG:2154)
# Coordinates can be found using online tools or GIS software. Example: Circa (651196, 6862551)
centre_paris_coords = (651196, 6862551)
centre_paris_point = Point(centre_paris_coords)

print(f"Defined center for Paris (Notre-Dame): {centre_paris_point}")

### Paris: 1. Employment Maps

These maps show the number of jobs in each commune of the Paris urban area for 1968, 1999, and 2021. Darker colors indicate a higher concentration of employment. We look for changes in the spatial pattern over time – specifically, a potential dispersion from the center or the emergence of secondary employment poles.

In [ ]:
def plot_map(gdf, year, city_name, output_dir):
    """Generates and displays/saves an employment map for a given year."""
    col_name = f'Total_Emploi_{year}'
    if gdf is None or col_name not in gdf.columns:
        print(f"Données manquantes pour la carte de {city_name} en {year}.")
        return

    print(f"Generating map for {city_name} - {year}...")

    # Use OrRd colormap
    cmap = plt.cm.OrRd

    # Normalize color scale based on employment range for this year
    vmin = gdf[col_name].min()
    vmax = gdf[col_name].max()
    # Handle potential case where min=max
    if vmin == vmax:
       vmin = vmax - 1 if vmax > 0 else 0
       vmax = vmax + 1 if vmax > 0 else 1
    norm = plt.Normalize(vmin=vmin, vmax=vmax)

    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    gdf.plot(
        column=col_name,
        cmap=cmap,
        norm=norm,
        linewidth=0.1, # Thin borders
        edgecolor='0.8', # Light grey borders
        legend=False, # We'll add a colorbar manually
        ax=ax,
        missing_kwds={'color': 'lightgrey'} # Color for missing values
    )

    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm._A = [] # Empty array for the data range
    cbar = fig.colorbar(sm, ax=ax, shrink=0.7) # Adjust shrink as needed
    cbar.set_label(f'Nombre d\'emplois ({year})')


    # Add center point to map
    if city_name=="Paris":
        gpd.GeoSeries([centre_paris_point], crs=gdf.crs).plot(ax=ax, marker='*', color='blue', markersize=100, label='Centre (Notre-Dame)')
        ax.legend()
    elif city_name=="Lyon":
         # Define Lyon center for plotting
         centre_lyon_point = Point(842072, 6519805)
         gpd.GeoSeries([centre_lyon_point], crs=gdf.crs).plot(ax=ax, marker='*', color='blue', markersize=100, label='Centre (Fourvière)')
         ax.legend()


    # Style and save
    ax.set_title(f"Nombre d'emplois par commune - Agglomération {city_name} en {year}")
    ax.set_axis_off() # Remove axis ticks and labels
    plt.tight_layout() # Adjust layout

    # Save the figure
    save_path = os.path.join(output_dir, "img/carte", f"carte_lieu_emplois_{city_name}_{year}.png")
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    print(f" Map saved to: {save_path}")

    # Display the plot inline
    plt.show()
    plt.close(fig) # Close the figure after showing/saving

# Generate maps for Paris for each period
if paris_com_gdf is not None:
    for year in PERIODES:
        plot_map(paris_com_gdf, year, "Paris", REPERTOIRE_SORTIE)
else:
    print("Skipping Paris maps because merged data is not available.")


**Interpretation (Paris Maps):**
*   **1968:** Observe the initial distribution. Is it strongly concentrated in the historical center (inner Paris)?
*   **1999:** Has the pattern changed? Are there signs of suburbanization of employment or the growth of specific areas like La Défense?
*   **2021:** What is the most recent pattern? Is it clearly monocentric, polycentric (with distinct subcenters), or dispersed? Compare the intensity of the center versus potential subcenters (e.g., La Défense, Saclay, Roissy).

---


### Paris: 2. Distance Calculation

We calculate the distance (in kilometers) from the centroid of each commune to the defined center of Paris (Notre-Dame). This is crucial for analyzing the spatial gradient.

In [ ]:
def calculer_distances(gdf, centre_point, city_name):
    """Calculates distance from commune centroid to a center point."""
    if gdf is None or 'X_CENTROID' not in gdf.columns or 'Y_CENTROID' not in gdf.columns:
        print(f"Données géographiques ou centroïdes manquants pour calculer les distances pour {city_name}.")
        return None

    print(f"Calculating distances from center for {city_name}...")

    # Ensure gdf has geometry and calculate distance
    # The distance is calculated in the units of the CRS (meters for Lambert 93)
    try:
        # Ensure 'geometry' column exists and try to access centroid
        if 'geometry' not in gdf.columns:
             print("ERREUR: Colonne 'geometry' manquante.")
             return gdf # Return original gdf if geometry missing

        # Calculate distance from the geometry's representative point (safer than centroid for complex shapes)
        gdf['distance_m'] = gdf.geometry.representative_point().distance(centre_point)
        gdf['distance_km'] = gdf['distance_m'] / 1000.0 # Convert to km

        print("Distances calculated successfully.")
        return gdf

    except Exception as e:
        print(f" Erreur lors du calcul des distances : {e}")
        # Attempt distance using pre-calculated centroids as fallback
        try:
            print(" Tentative de calcul via centroïdes pré-calculés...")
            gdf['distance_km'] = gdf.apply(
                 lambda row: Point(row["X_CENTROID"], row["Y_CENTROID"]).distance(centre_point) / 1000.0,
                 axis=1
            )
            print(" Distances calculées avec succès via centroïdes.")
            return gdf
        except Exception as e2:
            print(f" ERREUR: Échec du calcul des distances via centroïdes également: {e2}")
            return gdf # Return original gdf if distance fails


# Calculate distances for Paris
if paris_com_gdf is not None:
    paris_com_gdf = calculer_distances(paris_com_gdf, centre_paris_point, "Paris")
    if 'distance_km' in paris_com_gdf.columns:
        print("\nSample of Paris data with distances:")
        display(paris_com_gdf[['code', 'NOM_COM', 'distance_km'] + [f'Total_Emploi_{y}' for y in PERIODES]].head())
    else:
        print("Colonne 'distance_km' n'a pas pu être ajoutée pour Paris.")


### Paris: 3. Cumulative Employment Distribution

This plot shows the percentage of total metropolitan employment located within a given distance from the center. A steep curve indicates high central concentration, while a flatter curve suggests greater dispersion. We compare the curves for 1968, 1999, and 2021.

**Key Metric:** Percentage of jobs within 10 km of the center. A decrease over time suggests decentralization.

In [ ]:
def plot_cumulative_distribution(gdf, year, city_name, output_dir):
    """Plots the cumulative distribution of employment by distance."""
    col_name = f'Total_Emploi_{year}'
    dist_col = 'distance_km'

    if gdf is None or col_name not in gdf.columns or dist_col not in gdf.columns:
        print(f"Données manquantes pour la distribution cumulative de {city_name} en {year}.")
        return

    print(f"Generating cumulative distribution plot for {city_name} - {year}...")

    # Prepare data: sort by distance, calculate cumulative employment and percentage
    # Make sure to handle potential NaN distances
    df_analyse = gdf[[dist_col, col_name]].dropna(subset=[dist_col, col_name]).copy()
    if df_analyse.empty:
        print(f" AVERTISSEMENT: Aucune donnée valide pour {city_name} {year} après suppression des NaN.")
        return
        
    df_analyse = df_analyse.sort_values(dist_col)

    total_emplois = df_analyse[col_name].sum()
    if total_emplois <= 0:
        print(f" AVERTISSEMENT: Total d'emplois nul ou négatif pour {city_name} {year}. Skipping plot.")
        return

    df_analyse['emplois_cumules'] = df_analyse[col_name].cumsum()
    df_analyse['part_emplois_cumules'] = df_analyse['emplois_cumules'] / total_emplois

    # Calculate share within 10km
    df_10km = df_analyse[df_analyse[dist_col] <= 10]
    if not df_10km.empty:
        part_10km = df_10km['part_emplois_cumules'].max()
    else:
        part_10km = 0 # Handle case where no communes are within 10km
        
    if pd.isna(part_10km): part_10km = 0
    print(f" {city_name} {year}: {part_10km:.2%} des emplois sont à moins de 10 km du centre.")

    # Create plot
    plt.figure(figsize=(10, 6))
    plt.plot(df_analyse[dist_col], df_analyse['part_emplois_cumules'])
    plt.title(f"Répartition Cumulative Emplois {city_name} {year}")
    plt.xlabel("Distance (km)")
    plt.ylabel("Part Cumulée Emplois")
    plt.ylim(0, 1.05) # Set y-axis limits
    plt.grid(True)

    # Save plot
    save_path = os.path.join(output_dir, city_name, f"emplois_cumules_{city_name}_{year}.png")
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    print(f" Cumulative distribution plot saved to: {save_path}")

    # Display plot inline
    plt.show()
    plt.close() # Close the figure

# Generate cumulative plots for Paris
if paris_com_gdf is not None:
    for year in PERIODES:
        plot_cumulative_distribution(paris_com_gdf, year, "Paris", REPERTOIRE_SORTIE)
else:
    print("Skipping Paris cumulative distribution plots because data is not available.")


**Interpretation (Paris Cumulative Distribution):**
*   Compare the steepness of the curves for the different years. Did the curve become flatter over time, indicating employment spread further out?
*   Look at the percentage of jobs within 10 km. Did this percentage decrease significantly between 1968 and 2021? This is a strong indicator of decentralization. The presentation notes a drop from 67% (1968) to 54% (2021). Do our calculations confirm this?

---


### Paris: 4. Employment-Distance Relationship (Regression Analysis)

The standard monocentric model predicts that employment density decreases as distance from the center increases. We test this by plotting the natural logarithm of the number of jobs against the natural logarithm of the distance from the center. A negative slope is expected.

**Steps:**
1.  Filter out communes with zero distance or zero employment (logarithms are undefined).
2.  Create scatter plots of `log(Employment)` vs `log(Distance)`.
3.  Perform an Ordinary Least Squares (OLS) regression: `log(Employment) = β₀ + β₁ * log(Distance) + ε`.
4.  Analyze the results:
    *   **β₁ (Slope):** Should be negative. A steeper negative slope indicates faster decline of employment with distance (stronger central pull). A less steep slope might indicate polycentrism or other factors influencing location.
    *   **R²:** Indicates the proportion of the variance in log(Employment) explained by log(Distance). A higher R² suggests the simple distance model fits better.

In [ ]:
def analyser_relation_emploi_distance(gdf, year, city_name, output_dir):
    """Performs OLS regression of log(employment) vs log(distance) and plots the relationship."""
    col_name = f'Total_Emploi_{year}'
    dist_col = 'distance_km'

    if gdf is None or col_name not in gdf.columns or dist_col not in gdf.columns:
        print(f"Données manquantes pour l'analyse de régression de {city_name} en {year}.")
        return

    print(f"Performing regression analysis for {city_name} - {year}...")

    # Prepare data for regression
    # Filter out zero/negative distance and non-positive employment
    df_analyse = gdf[(gdf[dist_col] > 0) & (gdf[col_name] > 0)].copy()

    if df_analyse.empty:
        print(f" AVERTISSEMENT: Pas de données valides pour la régression ({city_name} {year}).")
        return

    # Calculate logarithms
    df_analyse['log_distance'] = np.log(df_analyse[dist_col])
    df_analyse['log_emploi'] = np.log(df_analyse[col_name])

    # Drop any NaNs/Infs that might result from log transformation or previous steps
    df_analyse.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_analyse = df_analyse.dropna(subset=['log_distance', 'log_emploi'])

    if len(df_analyse) < 2: # Need at least 2 points for regression
        print(f" AVERTISSEMENT: Pas assez de points de données ({len(df_analyse)}) pour la régression ({city_name} {year}).")
        return

    # Perform OLS regression
    try:
        formula = "log_emploi ~ log_distance"
        modele = smf.ols(formula, data=df_analyse).fit()

        # Display regression summary in the notebook output
        print(f"\n--- OLS Regression Results: {city_name} {year} ---")
        # Using display() for better formatting in Jupyter
        from IPython.display import display, HTML
        display(HTML(modele.summary().as_html()))
        print("------------------------------------------------\n")

        # Save summary to text file
        summary_path = os.path.join(output_dir, city_name, f"MCO_dist_emploi_{city_name}_{year}.txt")
        with open(summary_path, "w") as f:
            f.write(modele.summary().as_text())
        print(f" Regression summary saved to: {summary_path}")

        # Create scatter plot with regression line
        plt.figure(figsize=(10, 6))
        sns.regplot(
            x='log_distance',
            y='log_emploi',
            data=df_analyse,
            scatter_kws={'alpha':0.5},
            line_kws={'color': 'red'} # Make regression line stand out
        )
        plt.title(f"Relation Emploi-Distance {city_name} {year}")
        plt.xlabel("Log Distance (km)")
        plt.ylabel("Log Nombre d'Emplois")

        # Add R-squared to the plot legend or text
        plt.text(0.95, 0.95, f'$R^2 = {modele.rsquared:.2f}$',
                 ha='right', va='top', transform=plt.gca().transAxes,
                 bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

        plt.grid(True)

        # Save plot
        plot_path = os.path.join(output_dir, city_name, f"relation_emploi_{city_name}_{year}.png")
        plt.savefig(plot_path, dpi=200, bbox_inches='tight')
        print(f" Regression plot saved to: {plot_path}")

        # Display plot inline
        plt.show()
        plt.close() # Close the figure

    except Exception as e:
        print(f" ERREUR lors de l'exécution de la régression ou du traçage pour {city_name} {year}: {e}")

# Run regression analysis for Paris
if paris_com_gdf is not None:
    for year in PERIODES:
        analyser_relation_emploi_distance(paris_com_gdf, year, "Paris", REPERTOIRE_SORTIE)
else:
    print("Skipping Paris regression analysis because data is not available.")


**Interpretation (Paris Regression):**
*   **Slope (Coefficient of `log_distance`):** Confirm it's negative and statistically significant (P>|t| close to 0). Did the slope become less steep (closer to zero) between 1968 and 2021? A less steep slope suggests a weaker relationship between distance and employment concentration, potentially due to polycentrism.
*   **R²:** How well does the simple log-distance model explain employment variations? Did R² change over time? A decrease might suggest the monocentric model is becoming less accurate. The presentation notes R² values around 0.55-0.56, which are relatively stable.
*   **Visual Inspection:** Look at the scatter plot. Are there clusters of points significantly above the regression line at certain distances? These could represent subcenters. Does the scatter seem wider or more complex in later years?

---


## Analysis: Lyon

We repeat the same analysis steps for the Lyon metropolitan area.

**Center Definition:**
*   **Lyon Center:** Basilique Notre-Dame de Fourvière. We use its coordinates in the Lambert-93 projection (EPSG:2154).

In [ ]:
# Define Center for Lyon (Basilique Notre-Dame de Fourvière in Lambert-93 / EPSG:2154)
# Coordinates can be found using online tools or GIS software. Example: Circa (842072, 6519805)
centre_lyon_coords = (842072, 6519805)
centre_lyon_point = Point(centre_lyon_coords)

print(f"Defined center for Lyon (Fourvière): {centre_lyon_point}")

### Lyon: 1. Employment Maps

Visualize the spatial distribution of employment in Lyon for 1968, 1999, and 2021. Look for changes compared to Paris and the emergence of potential subcenters like Part-Dieu or Gerland.

In [ ]:
# Generate maps for Lyon for each period
if lyon_com_gdf is not None:
    for year in PERIODES:
        plot_map(lyon_com_gdf, year, "Lyon", REPERTOIRE_SORTIE)
else:
    print("Skipping Lyon maps because merged data is not available.")

**Interpretation (Lyon Maps):**
*   Compare Lyon's spatial structure to Paris'. Does it appear more or less centralized?
*   Identify the main historical center. How strong is its dominance over time?
*   Can you visually identify potential secondary centers like Part-Dieu (major business district developed later)? How do they compare to the main center?

---


### Lyon: 2. Distance Calculation

Calculate the distance of each commune's centroid to the defined center of Lyon (Fourvière).

In [ ]:
# Calculate distances for Lyon
if lyon_com_gdf is not None:
    lyon_com_gdf = calculer_distances(lyon_com_gdf, centre_lyon_point, "Lyon")
    if 'distance_km' in lyon_com_gdf.columns:
        print("\nSample of Lyon data with distances:")
        display(lyon_com_gdf[['code', 'NOM_COM', 'distance_km'] + [f'Total_Emploi_{y}' for y in PERIODES]].head())
    else:
         print("Colonne 'distance_km' n'a pas pu être ajoutée pour Lyon.")


### Lyon: 3. Cumulative Employment Distribution

Plot the cumulative share of employment as a function of distance from Lyon's center. Compare the curves over time and with those of Paris.

**Key Metric:** Percentage of jobs within 10 km.

In [ ]:
# Generate cumulative plots for Lyon
if lyon_com_gdf is not None:
    for year in PERIODES:
        plot_cumulative_distribution(lyon_com_gdf, year, "Lyon", REPERTOIRE_SORTIE)
else:
    print("Skipping Lyon cumulative distribution plots because data is not available.")

**Interpretation (Lyon Cumulative Distribution):**
*   Compare the steepness of Lyon's curves over time. Is there evidence of decentralization?
*   Compare Lyon's curves to Paris'. Is Lyon more or less centrally concentrated than Paris?
*   Analyze the percentage of jobs within 10 km. The presentation notes 94% (1968) vs 88% (2021), suggesting high concentration persists, although with a slight decrease. Do our results align?

---


### Lyon: 4. Employment-Distance Relationship (Regression Analysis)

Analyze the relationship between log(Employment) and log(Distance) for Lyon using scatter plots and OLS regression.

In [ ]:
# Run regression analysis for Lyon
if lyon_com_gdf is not None:
    for year in PERIODES:
        analyser_relation_emploi_distance(lyon_com_gdf, year, "Lyon", REPERTOIRE_SORTIE)
else:
    print("Skipping Lyon regression analysis because data is not available.")

**Interpretation (Lyon Regression):**
*   **Slope:** Is it negative and significant? Did it change significantly over time? Compare Lyon's slope to Paris'. Is Lyon's employment concentration more or less sensitive to distance? The presentation suggests a persistent strong gradient.
*   **R²:** How well does the simple model fit Lyon? Compare R² values over time and with Paris. The presentation notes R² around 0.39-0.47, lower than Paris, suggesting distance alone explains less of the employment pattern in Lyon, or perhaps the monocentric assumption fits less well even initially.
*   **Visual Inspection:** Examine the scatter plot for Lyon. Does it look different from Paris? Are subcenters less apparent or located differently relative to the main center?

---


## Comparative Synthesis & Conclusion

Let's summarize the findings based on the visual and quantitative analysis, drawing parallels with the presentation's conclusions.

**Paris:**
*   **Visuals (Maps):** Clear shift from a strongly monocentric pattern in 1968 towards a more complex, hierarchical polycentric structure by 2021. Strong secondary poles (like La Défense) are visible.
*   **Cumulative Distribution:** Significant decrease in central concentration (e.g., % jobs within 10km dropped from ~67% to ~54%). The curve flattened over time.
*   **Regression:** The negative relationship between log(employment) and log(distance) persists (R² stable around 0.55), but the slope likely became slightly less steep (though this needs careful checking of coefficients). The scatter might show more deviations in later years, hinting at the influence of subcenters.
*   **Conclusion for Paris:** Paris is **no longer strictly monocentric**. It exhibits a **confirmed hierarchical polycentric structure**, although the historical center remains very important.

**Lyon:**
*   **Visuals (Maps):** The structure remains more dominated by the historical center compared to Paris, even in 2021. While some secondary areas (Part-Dieu, Gerland) emerged, they appear less dominant relative to the center than Paris's subcenters.
*   **Cumulative Distribution:** High central concentration persists. The percentage of jobs within 10km remains very high (~88% in 2021), decreasing only slightly from 1968 (~94%). The curve remained relatively steep.
*   **Regression:** A significant negative relationship exists, but the R² is lower than Paris's (~0.4-0.5), suggesting the simple distance model captures less of the variation. The slope (gradient) appears strong and relatively stable over time, confirming persistent central influence.
*   **Conclusion for Lyon:** Lyon remains **largely monocentric**, although showing signs of **moderate polycentrism** with the emergence of limited secondary poles. The historical center retains strong dominance.

**Overall Answer to the Research Question:**
*   **"Are Paris and Lyon still monocentric agglomerations?"**
    *   **Paris: No.** It has evolved into a clear example of hierarchical polycentrism.
    *   **Lyon: Mostly yes.** While not perfectly monocentric due to emerging subcenters, its structure remains heavily dominated by the traditional center compared to Paris. It represents a case of moderate, less developed polycentrism.

---

## References

*   Alonso, W. (1964). *Location and Land Use*.
*   Baum-Snow, N. (2007). Did Highways Cause Suburbanization?.
*   Combes, P.-P. et al. (2019). The Costs of Agglomeration.
*   INSEE. (2010). Base permanente des équipements. [https://www.insee.fr/fr/statistiques/1893182](https://www.insee.fr/fr/statistiques/1893182)
*   INSEE. (2010). Recensement de la population. [https://www.insee.fr/fr/statistiques/1893185](https://www.insee.fr/fr/statistiques/1893185)
*   IGN. (2020). GEOFLA® - Base de données géographiques. [https://geoservices.ign.fr/geofla](https://geoservices.ign.fr/geofla)
*   Giuliano, G. & Small, K. A. (1991). Identification de sous-centres d'emploi à Los Angeles.
*   Mignot, D. & Aguilera, L. (2005). Évolution de la structure urbaine francilienne : de la monocentricité à une organisation plus complexe avec des pôles secondaires, tout en préservant la centralité parisienne.

---

**End of Notebook**